### Import lib

In [1]:
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import re

import pandas as pd
import numpy as np
import yfinance as yf
from tqdm.notebook import tqdm

### define paths

In [4]:
def find_project_root(start_path: Path) -> Path:
    """
    Finds project root by looking for README.md and config.yaml.
    Works whether notebook is run from repo root or notebooks/ folder.
    """
    start_path = start_path.resolve()

    for path in [start_path, *start_path.parents]:
        if (path / "README.md").exists() and (path / "config.yaml").exists():
            return path

    raise FileNotFoundError("Project root not found. Make sure README.md and config.yaml exist.")


PROJECT_ROOT = find_project_root(Path.cwd())

RAW_DIR = PROJECT_ROOT / "data" / "raw"
UNIVERSE_DIR = RAW_DIR / "universe"

YF_DIR = RAW_DIR / "yfinance"
STOCK_DIR = YF_DIR / "stocks"
INDEX_DIR = YF_DIR / "indices"
META_DIR = YF_DIR / "metadata"

STOCK_DIR.mkdir(parents=True, exist_ok=True)
INDEX_DIR.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)

UNIVERSE_PATH = UNIVERSE_DIR / "ind_nifty100list.csv"
VALIDATION_REPORT_PATH = META_DIR / "ticker_validation_report.csv"
FULL_DOWNLOAD_LOG_PATH = META_DIR / "full_download_log.csv"

print("Project root:", PROJECT_ROOT)
print("Universe path:", UNIVERSE_PATH)
print("Validation report path:", VALIDATION_REPORT_PATH)
print("Stock output dir:", STOCK_DIR)
print("Index output dir:", INDEX_DIR)

Project root: E:\Projects\marketguard-india
Universe path: E:\Projects\marketguard-india\data\raw\universe\ind_nifty100list.csv
Validation report path: E:\Projects\marketguard-india\data\raw\yfinance\metadata\ticker_validation_report.csv
Stock output dir: E:\Projects\marketguard-india\data\raw\yfinance\stocks
Index output dir: E:\Projects\marketguard-india\data\raw\yfinance\indices


### Read the data 

In [5]:
universe = pd.read_csv(UNIVERSE_PATH)
validation_report = pd.read_csv(VALIDATION_REPORT_PATH)

print("Universe shape:", universe.shape)
print("Validation report shape:", validation_report.shape)

display(universe.head())
display(validation_report.head())
display(validation_report["status"].value_counts())

Universe shape: (100, 9)
Validation report shape: (100, 11)


,symbol,yf_ticker,company_name,industry,source,snapshot_date,active_for_v1,series,isin_code
0,ABB,ABB.NS,ABB India Ltd.,Capital Goods,NSE/Nifty Indices current NIFTY 100 constituen...,2026-07-14,True,EQ,INE117A01022
1,ADANIENSOL,ADANIENSOL.NS,Adani Energy Solutions Ltd.,Power,NSE/Nifty Indices current NIFTY 100 constituen...,2026-07-14,True,EQ,INE931S01010
2,ADANIENT,ADANIENT.NS,Adani Enterprises Ltd.,Metals & Mining,NSE/Nifty Indices current NIFTY 100 constituen...,2026-07-14,True,EQ,INE423A01024
3,ADANIGREEN,ADANIGREEN.NS,Adani Green Energy Ltd.,Power,NSE/Nifty Indices current NIFTY 100 constituen...,2026-07-14,True,EQ,INE364U01010
4,ADANIPORTS,ADANIPORTS.NS,Adani Ports and Special Economic Zone Ltd.,Services,NSE/Nifty Indices current NIFTY 100 constituen...,2026-07-14,True,EQ,INE742F01042


,symbol,yf_ticker,company_name,industry,status,rows_returned,start_date,end_date,latest_close,has_ohlcv,issue
0,ABB,ABB.NS,ABB India Ltd.,Capital Goods,OK,1863,2019-01-01,2026-07-14,6892.000000,True,NaN
1,ADANIENSOL,ADANIENSOL.NS,Adani Energy Solutions Ltd.,Power,OK,1863,2019-01-01,2026-07-14,1690.300049,True,NaN
2,ADANIENT,ADANIENT.NS,Adani Enterprises Ltd.,Metals & Mining,OK,1863,2019-01-01,2026-07-14,3188.899902,True,NaN
3,ADANIGREEN,ADANIGREEN.NS,Adani Green Energy Ltd.,Power,OK,1863,2019-01-01,2026-07-14,1604.500000,True,NaN
4,ADANIPORTS,ADANIPORTS.NS,Adani Ports and Special Economic Zone Ltd.,Services,OK,1863,2019-01-01,2026-07-14,1821.900024,True,NaN


status
OK    100
Name: count, dtype: int64

### Validate the tickers in yfinance

In [6]:
valid_tickers = validation_report.loc[
    validation_report["status"] == "OK",
    ["symbol", "yf_ticker", "company_name", "industry"]
].copy()

valid_tickers = valid_tickers.drop_duplicates(subset=["yf_ticker"]).reset_index(drop=True)

print("Valid tickers:", len(valid_tickers))
display(valid_tickers.head())
display(valid_tickers.tail())

Valid tickers: 100


,symbol,yf_ticker,company_name,industry
0,ABB,ABB.NS,ABB India Ltd.,Capital Goods
1,ADANIENSOL,ADANIENSOL.NS,Adani Energy Solutions Ltd.,Power
2,ADANIENT,ADANIENT.NS,Adani Enterprises Ltd.,Metals & Mining
3,ADANIGREEN,ADANIGREEN.NS,Adani Green Energy Ltd.,Power
4,ADANIPORTS,ADANIPORTS.NS,Adani Ports and Special Economic Zone Ltd.,Services


,symbol,yf_ticker,company_name,industry
95,UNITDSPR,UNITDSPR.NS,United Spirits Ltd.,Fast Moving Consumer Goods
96,VBL,VBL.NS,Varun Beverages Ltd.,Fast Moving Consumer Goods
97,VEDL,VEDL.NS,Vedanta Ltd.,Metals & Mining
98,WIPRO,WIPRO.NS,Wipro Ltd.,Information Technology
99,ZYDUSLIFE,ZYDUSLIFE.NS,Zydus Lifesciences Ltd.,Healthcare


### set the parameters 

In [8]:
START_DATE = "2010-01-01"
END_DATE = None
INTERVAL = "1d"

MAX_WORKERS = 8
MAX_RETRIES = 5
RETRY_SLEEP_SECONDS = 2

print("Start date:", START_DATE)
print("End date:", END_DATE)
print("Interval:", INTERVAL)
print("Max workers:", MAX_WORKERS)
print("Max retries:", MAX_RETRIES)

Start date: 2010-01-01
End date: None
Interval: 1d
Max workers: 8
Max retries: 5


### Helper function to standardize yfinance data

In [9]:
def safe_file_name(ticker: str) -> str:
    """
    Converts ticker like TCS.NS or ^NSEI into safe file name.
    """
    name = ticker.replace("^", "")
    name = name.replace(".", "_")
    name = re.sub(r"[^A-Za-z0-9_\-]", "_", name)
    return name


def flatten_yfinance_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    yfinance sometimes returns MultiIndex columns.
    For single ticker downloads, the second level is usually ticker name.
    We flatten safely while preserving all available returned columns.
    """
    df = df.copy()

    if isinstance(df.columns, pd.MultiIndex):
        new_cols = []

        for col in df.columns:
            parts = [str(x).strip() for x in col if str(x).strip() not in ["", "nan", "None"]]

            if len(parts) == 0:
                new_cols.append("unknown")
            elif len(parts) == 1:
                new_cols.append(parts[0])
            else:
                # Usually ('Close', 'TCS.NS').
                # Keep first part if second part looks like ticker.
                # Otherwise join both parts.
                if parts[1].endswith(".NS") or parts[1].startswith("^"):
                    new_cols.append(parts[0])
                else:
                    new_cols.append("_".join(parts))

        df.columns = new_cols

    return df


def standardize_raw_yfinance_df(
    df: pd.DataFrame,
    symbol: str,
    yf_ticker: str,
    company_name: str = None,
    industry: str = None,
    asset_type: str = "stock",
) -> pd.DataFrame:
    """
    Preserve every column returned by yfinance.
    Only standardizes:
    - date column
    - metadata columns
    - snake_case column names
    - row ordering
    """

    if df.empty:
        return df

    df = flatten_yfinance_columns(df)
    df = df.reset_index()

    if "Date" in df.columns:
        date_col = "Date"
    elif "Datetime" in df.columns:
        date_col = "Datetime"
    else:
        raise ValueError(f"No Date/Datetime column found for {yf_ticker}")

    df[date_col] = pd.to_datetime(df[date_col]).dt.date

    # Add metadata columns. These are useful later and do not remove any yfinance columns.
    df.insert(0, "asset_type", asset_type)
    df.insert(1, "symbol", symbol)
    df.insert(2, "yf_ticker", yf_ticker)

    if company_name is not None:
        df.insert(3, "company_name", company_name)

    if industry is not None:
        df.insert(4, "industry", industry)

    # Clean column names after adding metadata.
    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace(".", "", regex=False)
        .str.replace("-", "_", regex=False)
    )

    if "datetime" in df.columns:
        df = df.rename(columns={"datetime": "date"})

    if "date" not in df.columns:
        raise ValueError(f"Date column standardization failed for {yf_ticker}")

    df = df.sort_values("date")
    df = df.drop_duplicates(subset=["date", "yf_ticker"], keep="last")

    return df


def download_from_yfinance(yf_ticker: str) -> pd.DataFrame:
    """
    Downloads raw daily EOD data from yfinance.
    actions=True tries to include dividends and stock splits.
    auto_adjust=False keeps raw OHLC and adjusted close separately.
    threads=False because we are using ThreadPoolExecutor outside.
    """
    return yf.download(
        yf_ticker,
        start=START_DATE,
        end=END_DATE,
        interval=INTERVAL,
        auto_adjust=False,
        actions=True,
        progress=False,
        threads=False,
        repair=True,
    )

### Download one asset with retries

In [10]:
def download_and_save_one_asset(row: dict, output_dir: Path, asset_type: str) -> dict:
    """
    Downloads one ticker, preserves all returned yfinance columns,
    saves CSV, and returns one log row.
    """

    symbol = row.get("symbol")
    yf_ticker = row.get("yf_ticker")
    company_name = row.get("company_name")
    industry = row.get("industry")

    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            raw_df = download_from_yfinance(yf_ticker)

            if raw_df.empty:
                last_error = "No data returned"
                time.sleep(RETRY_SLEEP_SECONDS * attempt)
                continue

            clean_df = standardize_raw_yfinance_df(
                df=raw_df,
                symbol=symbol,
                yf_ticker=yf_ticker,
                company_name=company_name,
                industry=industry,
                asset_type=asset_type,
            )

            safe_name = safe_file_name(yf_ticker)
            output_path = output_dir / f"{safe_name}.csv"
            clean_df.to_csv(output_path, index=False)

            return {
                "symbol": symbol,
                "yf_ticker": yf_ticker,
                "asset_type": asset_type,
                "status": "OK",
                "rows": len(clean_df),
                "start_date": clean_df["date"].min(),
                "end_date": clean_df["date"].max(),
                "columns_returned": "|".join(clean_df.columns.tolist()),
                "file_path": str(output_path.relative_to(PROJECT_ROOT)),
                "attempts": attempt,
                "issue": "",
                "downloaded_at": datetime.now().isoformat(timespec="seconds"),
            }

        except Exception as e:
            last_error = str(e)
            time.sleep(RETRY_SLEEP_SECONDS * attempt)

    return {
        "symbol": symbol,
        "yf_ticker": yf_ticker,
        "asset_type": asset_type,
        "status": "ERROR",
        "rows": 0,
        "start_date": None,
        "end_date": None,
        "columns_returned": "",
        "file_path": None,
        "attempts": MAX_RETRIES,
        "issue": last_error,
        "downloaded_at": datetime.now().isoformat(timespec="seconds"),
    }

### Test one stock

In [14]:
test_row = valid_tickers.loc[
    valid_tickers["yf_ticker"] == "WIPRO.NS"
].iloc[0].to_dict()

test_result = download_and_save_one_asset(
    row=test_row,
    output_dir=STOCK_DIR,
    asset_type="stock",
)

test_result

{'symbol': 'WIPRO',
 'yf_ticker': 'WIPRO.NS',
 'asset_type': 'stock',
 'status': 'OK',
 'rows': 4082,
 'start_date': datetime.date(2010, 1, 4),
 'end_date': datetime.date(2026, 7, 14),
 'columns_returned': 'asset_type|symbol|yf_ticker|company_name|industry|date|adj_close|close|dividends|high|low|open|repaired?|stock_splits|volume',
 'file_path': 'data\\raw\\yfinance\\stocks\\WIPRO_NS.csv',
 'attempts': 1,
 'issue': '',
 'downloaded_at': '2026-07-14T16:23:03'}

In [15]:
test_path = PROJECT_ROOT / test_result["file_path"]

test_df = pd.read_csv(test_path)

print(test_path)
print(test_df.shape)
display(test_df.head())
display(test_df.tail())
print(test_df.columns.tolist())

E:\Projects\marketguard-india\data\raw\yfinance\stocks\WIPRO_NS.csv
(4082, 15)


,asset_type,symbol,yf_ticker,company_name,industry,date,adj_close,close,dividends,high,low,open,repaired?,stock_splits,volume
0,stock,WIPRO,WIPRO.NS,Wipro Ltd.,Information Technology,2010-01-04,63.930515,78.052505,0.0,78.412506,76.044380,77.062500,False,0.0,6819252
1,stock,WIPRO,WIPRO.NS,Wipro Ltd.,Information Technology,2010-01-05,64.939537,79.284378,0.0,79.650002,78.024376,78.333755,False,0.0,9959403
2,stock,WIPRO,WIPRO.NS,Wipro Ltd.,Information Technology,2010-01-06,63.626442,77.681252,0.0,79.875000,77.287506,79.425003,False,0.0,9150061
3,stock,WIPRO,WIPRO.NS,Wipro Ltd.,Information Technology,2010-01-07,62.497677,76.303131,0.0,78.243752,75.937500,78.187500,False,0.0,9165297
4,stock,WIPRO,WIPRO.NS,Wipro Ltd.,Information Technology,2010-01-08,61.557785,75.155624,0.0,76.500000,74.925003,76.387505,False,0.0,5713679


,asset_type,symbol,yf_ticker,company_name,industry,date,adj_close,close,dividends,high,low,open,repaired?,stock_splits,volume
4077,stock,WIPRO,WIPRO.NS,Wipro Ltd.,Information Technology,2026-07-08,172.720001,172.720001,0.0,175.399994,172.240005,173.000000,False,0.0,16482564
4078,stock,WIPRO,WIPRO.NS,Wipro Ltd.,Information Technology,2026-07-09,172.759995,172.759995,0.0,174.289993,171.649994,173.000000,False,0.0,10270117
4079,stock,WIPRO,WIPRO.NS,Wipro Ltd.,Information Technology,2026-07-10,175.460007,175.460007,0.0,177.220001,175.000000,175.229996,False,0.0,16249590
4080,stock,WIPRO,WIPRO.NS,Wipro Ltd.,Information Technology,2026-07-13,178.429993,178.429993,0.0,179.949997,174.360001,174.600006,False,0.0,22417855
4081,stock,WIPRO,WIPRO.NS,Wipro Ltd.,Information Technology,2026-07-14,177.139999,177.139999,0.0,180.500000,176.710007,178.500000,False,0.0,14457481


['asset_type', 'symbol', 'yf_ticker', 'company_name', 'industry', 'date', 'adj_close', 'close', 'dividends', 'high', 'low', 'open', 'repaired?', 'stock_splits', 'volume']


### Parallel download all NIFTY 100 stocks

In [16]:
stock_rows = valid_tickers.to_dict(orient="records")

stock_download_logs = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(
            download_and_save_one_asset,
            row,
            STOCK_DIR,
            "stock",
        ): row
        for row in stock_rows
    }

    for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading stocks"):
        result = future.result()
        stock_download_logs.append(result)

stock_download_log = pd.DataFrame(stock_download_logs)
stock_download_log = stock_download_log.sort_values("symbol").reset_index(drop=True)

display(stock_download_log.head())
display(stock_download_log["status"].value_counts())

,symbol,yf_ticker,asset_type,status,rows,start_date,end_date,columns_returned,file_path,attempts,issue,downloaded_at
0,ABB,ABB.NS,stock,OK,4082,2010-01-04,2026-07-14,asset_type|symbol|yf_ticker|company_name|indus...,data\raw\yfinance\stocks\ABB_NS.csv,1,,2026-07-14T16:26:03
1,ADANIENSOL,ADANIENSOL.NS,stock,OK,2705,2015-07-31,2026-07-14,asset_type|symbol|yf_ticker|company_name|indus...,data\raw\yfinance\stocks\ADANIENSOL_NS.csv,1,,2026-07-14T16:26:02
2,ADANIENT,ADANIENT.NS,stock,OK,4082,2010-01-04,2026-07-14,asset_type|symbol|yf_ticker|company_name|indus...,data\raw\yfinance\stocks\ADANIENT_NS.csv,1,,2026-07-14T16:26:03
3,ADANIGREEN,ADANIGREEN.NS,stock,OK,1995,2018-06-18,2026-07-14,asset_type|symbol|yf_ticker|company_name|indus...,data\raw\yfinance\stocks\ADANIGREEN_NS.csv,1,,2026-07-14T16:26:02
4,ADANIPORTS,ADANIPORTS.NS,stock,OK,4082,2010-01-04,2026-07-14,asset_type|symbol|yf_ticker|company_name|indus...,data\raw\yfinance\stocks\ADANIPORTS_NS.csv,1,,2026-07-14T16:26:02


status
OK    100
Name: count, dtype: int64

### Inspect failed stock downloads

In [17]:
failed_stocks = stock_download_log[
    stock_download_log["status"] != "OK"
].copy()

print("Failed/errored stocks:", len(failed_stocks))
display(failed_stocks)

Failed/errored stocks: 0


,symbol,yf_ticker,asset_type,status,rows,start_date,end_date,columns_returned,file_path,attempts,issue,downloaded_at


# Index data download 

### defining path

In [27]:
INDEX_UNIVERSE_DIR = RAW_DIR / "index_universe"
INDEX_UNIVERSE_DIR.mkdir(parents=True, exist_ok=True)

CORE_INDICES_V1_FINAL_PATH = INDEX_UNIVERSE_DIR / "core_indices_v1_final.csv"

print("Core index file:", CORE_INDICES_V1_FINAL_PATH)

Core index file: E:\Projects\marketguard-india\data\raw\index_universe\core_indices_v1_final.csv


### Load final approved indices

In [28]:
core_indices = pd.read_csv(CORE_INDICES_V1_FINAL_PATH)

print("Core indices shape:", core_indices.shape)
display(core_indices)

Core indices shape: (16, 4)


,purpose,index_name,yf_ticker,use_for_v1
0,Market,Nifty 50,^NSEI,Yes
1,Market,Nifty 100,^CNX100,Yes
2,Volatility,India VIX,^INDIAVIX,Yes
3,Banking,Nifty Bank,^NSEBANK,Yes
4,Auto,Nifty Auto,^CNXAUTO,Yes
5,IT,Nifty IT,^CNXIT,Yes
6,Energy,Nifty Energy,^CNXENERGY,Yes
7,FMCG,Nifty FMCG,^CNXFMCG,Yes
8,Metal,Nifty Metal,^CNXMETAL,Yes
9,Pharma,Nifty Pharma,^CNXPHARMA,Yes


### Prepare index rows for download

In [29]:
index_tickers_for_download = core_indices.copy()

index_tickers_for_download = index_tickers_for_download.rename(columns={
    "index_name": "symbol",
    "purpose": "industry",
})

index_tickers_for_download["company_name"] = index_tickers_for_download["symbol"]

index_tickers_for_download = index_tickers_for_download[
    ["symbol", "yf_ticker", "company_name", "industry"]
].copy()

display(index_tickers_for_download)
print("Index tickers for download:", len(index_tickers_for_download))

,symbol,yf_ticker,company_name,industry
0,Nifty 50,^NSEI,Nifty 50,Market
1,Nifty 100,^CNX100,Nifty 100,Market
2,India VIX,^INDIAVIX,India VIX,Volatility
3,Nifty Bank,^NSEBANK,Nifty Bank,Banking
4,Nifty Auto,^CNXAUTO,Nifty Auto,Auto
5,Nifty IT,^CNXIT,Nifty IT,IT
6,Nifty Energy,^CNXENERGY,Nifty Energy,Energy
7,Nifty FMCG,^CNXFMCG,Nifty FMCG,FMCG
8,Nifty Metal,^CNXMETAL,Nifty Metal,Metal
9,Nifty Pharma,^CNXPHARMA,Nifty Pharma,Pharma


Index tickers for download: 16


### Validate the index tickers before full download

In [31]:
def validate_yf_ticker(yf_ticker: str, start_date: str = "2015-01-01") -> dict:
    """
    Quick yfinance validation for stocks/indices.
    Checks whether ticker returns usable daily OHLCV-style data.
    """

    try:
        df = yf.download(
            yf_ticker,
            start=start_date,
            interval="1d",
            auto_adjust=False,
            actions=True,
            progress=False,
            threads=False,
            repair=True,
        )

        if df.empty:
            return {
                "status": "FAILED",
                "rows_returned": 0,
                "start_date": None,
                "end_date": None,
                "latest_close": np.nan,
                "has_ohlcv": False,
                "issue": "No data returned",
            }

        # Handle yfinance MultiIndex columns
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = [
                col[0] if isinstance(col, tuple) else col
                for col in df.columns
            ]

        df = df.reset_index()

        date_col = None
        if "Date" in df.columns:
            date_col = "Date"
        elif "Datetime" in df.columns:
            date_col = "Datetime"

        if date_col is None:
            return {
                "status": "FAILED",
                "rows_returned": len(df),
                "start_date": None,
                "end_date": None,
                "latest_close": np.nan,
                "has_ohlcv": False,
                "issue": "No Date/Datetime column",
            }

        required_cols = ["Open", "High", "Low", "Close"]
        missing_cols = [col for col in required_cols if col not in df.columns]

        if missing_cols:
            return {
                "status": "FAILED",
                "rows_returned": len(df),
                "start_date": df[date_col].min(),
                "end_date": df[date_col].max(),
                "latest_close": np.nan,
                "has_ohlcv": False,
                "issue": f"Missing columns: {missing_cols}",
            }

        price_cols = ["Open", "High", "Low", "Close"]
        bad_prices = (df[price_cols] <= 0).sum().sum()

        status = "OK"
        issue = ""

        if bad_prices > 0:
            status = "FLAGGED"
            issue = f"Non-positive price values found: {bad_prices}"

        latest_close = (
            df["Close"].dropna().iloc[-1]
            if "Close" in df.columns and df["Close"].notna().any()
            else np.nan
        )

        return {
            "status": status,
            "rows_returned": len(df),
            "start_date": df[date_col].min(),
            "end_date": df[date_col].max(),
            "latest_close": latest_close,
            "has_ohlcv": True,
            "issue": issue,
        }

    except Exception as e:
        return {
            "status": "ERROR",
            "rows_returned": 0,
            "start_date": None,
            "end_date": None,
            "latest_close": np.nan,
            "has_ohlcv": False,
            "issue": str(e),
        }

In [32]:
index_validation_rows = []

for _, row in tqdm(index_tickers_for_download.iterrows(), total=len(index_tickers_for_download)):
    result = validate_yf_ticker(row["yf_ticker"])

    index_validation_rows.append({
        "symbol": row["symbol"],
        "yf_ticker": row["yf_ticker"],
        "company_name": row["company_name"],
        "industry": row["industry"],
        **result,
    })

    time.sleep(0.15)

index_validation_report = pd.DataFrame(index_validation_rows)

display(index_validation_report)
display(index_validation_report["status"].value_counts())

  0%|          | 0/16 [00:00<?, ?it/s]

,symbol,yf_ticker,company_name,industry,status,rows_returned,start_date,end_date,latest_close,has_ohlcv,issue
0,Nifty 50,^NSEI,Nifty 50,Market,OK,2837,2015-01-02,2026-07-14,24052.050781,True,
1,Nifty 100,^CNX100,Nifty 100,Market,OK,2839,2015-01-01,2026-07-14,25088.650391,True,
2,India VIX,^INDIAVIX,India VIX,Volatility,OK,2829,2015-01-01,2026-07-14,13.750000,True,
3,Nifty Bank,^NSEBANK,Nifty Bank,Banking,OK,2843,2015-01-01,2026-07-14,57462.300781,True,
4,Nifty Auto,^CNXAUTO,Nifty Auto,Auto,OK,2830,2015-01-01,2026-07-14,26547.500000,True,
5,Nifty IT,^CNXIT,Nifty IT,IT,OK,2843,2015-01-01,2026-07-14,28724.750000,True,
6,Nifty Energy,^CNXENERGY,Nifty Energy,Energy,OK,2830,2015-01-01,2026-07-14,39203.148438,True,
7,Nifty FMCG,^CNXFMCG,Nifty FMCG,FMCG,OK,2829,2015-01-01,2026-07-14,48524.949219,True,
8,Nifty Metal,^CNXMETAL,Nifty Metal,Metal,OK,2830,2015-01-01,2026-07-14,12677.700195,True,
9,Nifty Pharma,^CNXPHARMA,Nifty Pharma,Pharma,OK,2843,2015-01-01,2026-07-14,25907.099609,True,


status
OK    16
Name: count, dtype: int64

### Use only valid indices for download

In [33]:
valid_index_tickers = index_validation_report[
    index_validation_report["status"] == "OK"
][["symbol", "yf_ticker", "company_name", "industry"]].copy()

print("Valid index tickers:", len(valid_index_tickers))
display(valid_index_tickers)

Valid index tickers: 16


,symbol,yf_ticker,company_name,industry
0,Nifty 50,^NSEI,Nifty 50,Market
1,Nifty 100,^CNX100,Nifty 100,Market
2,India VIX,^INDIAVIX,India VIX,Volatility
3,Nifty Bank,^NSEBANK,Nifty Bank,Banking
4,Nifty Auto,^CNXAUTO,Nifty Auto,Auto
5,Nifty IT,^CNXIT,Nifty IT,IT
6,Nifty Energy,^CNXENERGY,Nifty Energy,Energy
7,Nifty FMCG,^CNXFMCG,Nifty FMCG,FMCG
8,Nifty Metal,^CNXMETAL,Nifty Metal,Metal
9,Nifty Pharma,^CNXPHARMA,Nifty Pharma,Pharma


### Download valid indices in parallel

In [34]:
index_rows = valid_index_tickers.to_dict(orient="records")

index_download_logs = []

with ThreadPoolExecutor(max_workers=min(MAX_WORKERS, len(index_rows))) as executor:
    futures = {
        executor.submit(
            download_and_save_one_asset,
            row,
            INDEX_DIR,
            "index",
        ): row
        for row in index_rows
    }

    for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading indices"):
        result = future.result()
        index_download_logs.append(result)

index_download_log = pd.DataFrame(index_download_logs)
index_download_log = index_download_log.sort_values("symbol").reset_index(drop=True)

display(index_download_log)
display(index_download_log["status"].value_counts())

,symbol,yf_ticker,asset_type,status,rows,start_date,end_date,columns_returned,file_path,attempts,issue,downloaded_at
0,India VIX,^INDIAVIX,index,OK,4060,2010-01-04,2026-07-14,asset_type|symbol|yf_ticker|company_name|indus...,data\raw\yfinance\indices\INDIAVIX.csv,1,,2026-07-14T23:18:21
1,Nifty 100,^CNX100,index,OK,4070,2010-01-04,2026-07-14,asset_type|symbol|yf_ticker|company_name|indus...,data\raw\yfinance\indices\CNX100.csv,1,,2026-07-14T23:18:21
2,Nifty 50,^NSEI,index,OK,4058,2010-01-04,2026-07-14,asset_type|symbol|yf_ticker|company_name|indus...,data\raw\yfinance\indices\NSEI.csv,1,,2026-07-14T23:18:21
3,Nifty Auto,^CNXAUTO,index,OK,3680,2011-07-12,2026-07-14,asset_type|symbol|yf_ticker|company_name|indus...,data\raw\yfinance\indices\CNXAUTO.csv,1,,2026-07-14T23:18:19
4,Nifty Bank,^NSEBANK,index,OK,4074,2010-01-04,2026-07-14,asset_type|symbol|yf_ticker|company_name|indus...,data\raw\yfinance\indices\NSEBANK.csv,1,,2026-07-14T23:18:20
5,Nifty Commodities,^CNXCMDT,index,OK,3641,2011-09-07,2026-07-14,asset_type|symbol|yf_ticker|company_name|indus...,data\raw\yfinance\indices\CNXCMDT.csv,1,,2026-07-14T23:18:26
6,Nifty Energy,^CNXENERGY,index,OK,3792,2011-01-31,2026-07-14,asset_type|symbol|yf_ticker|company_name|indus...,data\raw\yfinance\indices\CNXENERGY.csv,1,,2026-07-14T23:18:21
7,Nifty FMCG,^CNXFMCG,index,OK,3791,2011-01-31,2026-07-14,asset_type|symbol|yf_ticker|company_name|indus...,data\raw\yfinance\indices\CNXFMCG.csv,1,,2026-07-14T23:18:21
8,Nifty IT,^CNXIT,index,OK,4074,2010-01-04,2026-07-14,asset_type|symbol|yf_ticker|company_name|indus...,data\raw\yfinance\indices\CNXIT.csv,1,,2026-07-14T23:18:21
9,Nifty MNC,^CNXMNC,index,OK,3792,2011-01-31,2026-07-14,asset_type|symbol|yf_ticker|company_name|indus...,data\raw\yfinance\indices\CNXMNC.csv,1,,2026-07-14T23:18:26


status
OK    16
Name: count, dtype: int64